> **Data availability:** The input file `raman_preprocessed_intensity_matrix_integer_wavenumbers_400_3500.xlsx` is not included in this repository. The resulting functional dataset is provided (aligned) as `data/raman_functional_intensity_matrix_bspline_n4_aligned_402_3499.xlsx`. This notebook is included to document the processing steps; its file paths refer to the original project layout.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import BSpline, make_lsq_spline

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

## 1. Load the Discrete Intensity Matrix

In [ ]:
IN_PATH = 'Final/raman_preprocessed_intensity_matrix_integer_wavenumbers_400_3500.xlsx'

raw_matrix = pd.read_excel(IN_PATH, sheet_name='Intensity Matrix', index_col=0)
print(f'Loaded intensity matrix: {raw_matrix.shape[0]} samples x {raw_matrix.shape[1]} wavenumber points')

# Orient wavenumber-on-index, sample-per-column — consistent with the other notebooks
discrete_df = raw_matrix.T
discrete_df.index.name = 'Raman Shift'
discrete_df.index = discrete_df.index.astype(float)

x = discrete_df.index.values.astype(float)
print(f'Wavenumber range: {x.min():.0f} - {x.max():.0f} cm-1, {len(x)} points, spacing = {np.diff(x).mean():.2f} cm-1')
print(f'Samples: {list(discrete_df.columns)}')

## 2. B-spline Basis Expansion (order n = 4, cubic)

Each discrete spectrum is projected onto a cubic B-spline basis (order n = 4, degree k = 3) using a least-squares spline fit (`scipy.interpolate.make_lsq_spline`), with interior knots placed at every 10th wavenumber point.

In [ ]:
n = 4          # spline order (order 4 = cubic)
k = n - 1      # spline degree = 3

# Interior knots: every 10th point, excluding the boundary
interior_knots = x[10:-10:10]
t = np.concatenate(([x[0]] * (k + 1), interior_knots, [x[-1]] * (k + 1)))

functional_coeffs = {}   # spline coefficients per spectrum
functional_curves = {}   # spline-evaluated curve per spectrum, on the original grid

for col in discrete_df.columns:
    y = discrete_df[col].values
    spl = make_lsq_spline(x, y, t=t, k=k)
    functional_coeffs[col] = spl.c
    functional_curves[col] = spl(x)

functional_df = pd.DataFrame(functional_curves, index=discrete_df.index)
functional_df.index.name = 'Raman Shift'

print(f'B-spline basis: order n = {n} (degree k = {k})')
print(f'Interior knots : {len(interior_knots)}  ->  total basis functions = {len(t) - k - 1}')
print(f'Functional data shape: {functional_df.shape}')

## 3. Fit Quality — Discrete vs Functional

In [ ]:
max_dev = np.max(np.abs(functional_df.values - discrete_df.values), axis=0)
rmse = np.sqrt(np.mean((functional_df.values - discrete_df.values) ** 2, axis=0))

fit_quality = pd.DataFrame({
    'Max abs deviation': max_dev,
    'RMSE': rmse,
}, index=discrete_df.columns)

print(f'Max abs deviation (spline vs discrete): mean={max_dev.mean():.2e}, max={max_dev.max():.2e}')
print(f'RMSE (spline vs discrete): mean={rmse.mean():.2e}, max={rmse.max():.2e}')
fit_quality.head()

## 4. Visual Check — Discrete vs B-spline Fit

In [ ]:
examples = list(discrete_df.columns[:3])
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, col in zip(axes, examples):
    ax.plot(x, discrete_df[col].values, alpha=0.5, linewidth=0.8, color='steelblue', label='Discrete')
    ax.plot(x, functional_df[col].values, linewidth=1.2, color='crimson', label='B-spline fit')
    ax.set_title(col, fontsize=8)
    ax.set_xlabel('Raman Shift (cm⁻¹)')
axes[0].set_ylabel('Normalised Intensity (a.u.)')
axes[0].legend(fontsize=8)
plt.suptitle('Functional Representation: B-spline Fit vs Discrete Intensity Matrix', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Save Functional Intensity Matrix

In [ ]:
OUT_PATH = 'Final/raman_functional_intensity_matrix_bspline_n4.xlsx'

# Match the input layout: rows = sample names, columns = integer wavenumbers
functional_out = functional_df.T
functional_out.index.name = 'Sample Name'

readme = pd.DataFrame({
    'Value': [
        'Rows are sample names; columns are integer Raman shift wavenumbers in cm^-1; cells are B-spline functional intensities.',
        IN_PATH,
        f'{len(x)}-point integer grid from {x.min():.0f} to {x.max():.0f} cm^-1.',
        f'Cubic B-spline basis expansion, order n = {n} (degree k = {k}), fit via scipy.interpolate.make_lsq_spline.',
        'Interior knots placed at every 10th wavenumber point, matching raman_consensus.ipynb / raman_adulterated_dataset.ipynb / raman_evaluation_dataset.ipynb.',
        f'{len(interior_knots)} interior knots, {len(t) - k - 1} total basis functions.',
        functional_out.shape[0],
        functional_out.shape[1],
        f'{max_dev.mean():.2e} (mean), {max_dev.max():.2e} (max)',
    ],
}, index=[
    'Output layout',
    'Source data',
    'Source grid',
    'B-spline settings',
    'Knot placement',
    'Basis functions',
    'Samples',
    'Wavenumber columns',
    'Max abs deviation (spline vs discrete)',
])
readme.index.name = 'Field'

with pd.ExcelWriter(OUT_PATH) as writer:
    functional_out.to_excel(writer, sheet_name='Functional Intensity Matrix')
    readme.to_excel(writer, sheet_name='README')

print(f'Saved -> {OUT_PATH}')
print(f'  Sheet "Functional Intensity Matrix": {functional_out.shape[0]} samples x {functional_out.shape[1]} wavenumber points')